# Belebele E4B 8-bit: text versus rendered images

Analyze the completed **E4B INT8 run: 900 IDs × 12 languages × 2 modalities** in `results/belebele-e4b-8bit-full`. Run all cells using the project `.venv` kernel. All statistics and plots come from this run only; no earlier run is loaded or compared.

This notebook reads existing JSON results only: **no model loading, inference, dataset downloads, rendering, or image files**. It replays strict scores and reports the measured results. The saved notebook starts clean; plots appear when cells run. Clear outputs before saving/sharing if you want to keep it that way.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from rendering_eval.reporting import (
    build_report, LANGUAGE_NAMES, plot_overview, plot_answer_bias, plot_diagnostics,
)

RESULTS_DIR = Path("results/belebele-e4b-8bit-full")
report = build_report(RESULTS_DIR)
if report["status"] != "complete":
    raise ValueError("Select a completed evaluation before using this notebook.")
print("Run:", RESULTS_DIR)
print("Model:", report["experiment"]["model_id"], "—", report["experiment"]["quantization"])
plot_handles = []
def show_plot(figure):
    plot_handles.append(display(figure, display_id=True))
    plt.close(figure)

print(f"Status: {report['status']}; {report['audit']['predictions']:,}/"
      f"{report['audit']['expected_predictions']:,} predictions")
print(f"Strict scoring replayed; {report['audit']['duplicates']} duplicates; "
      f"{report['audit']['execution_error_attempts']} execution errors")
print(f"{report['audit']['unique_questions']} aligned questions from "
      f"{report['audit']['unique_passages']} passages; "
      f"{report['audit']['paired_examples']:,} text/image pairs")

## Overall result and language comparisons

Text and image conditions contain the same passage, question, and ordered answer choices. Invalid or truncated responses count as incorrect. The gap is **image minus text**, in percentage points; a negative gap means lower image accuracy.

In [ ]:
overall = report["groups"][0]
display(Markdown(
    f"**Text: {100 * overall['text_accuracy']:.2f}% · "
    f"Image: {100 * overall['image_accuracy']:.2f}% · "
    f"Image minus text: {overall['image_minus_text_pp']:.2f} percentage points**"
))
show_plot(plot_overview(report))

language_table = pd.DataFrame(report["groups"][1:])
language_table["Language"] = language_table.language.map(LANGUAGE_NAMES)
language_table["Text %"] = 100 * language_table.text_accuracy
language_table["Image %"] = 100 * language_table.image_accuracy
language_table["Gap (pp)"] = language_table.image_minus_text_pp
display(language_table[["Language", "paired_examples", "Text %", "Image %", "Gap (pp)", "invalid_responses"]].style.format(
    {"Text %": "{:.2f}", "Image %": "{:.2f}", "Gap (pp)": "{:.2f}"}))

In [ ]:
n = overall["paired_examples"]
outcomes = pd.DataFrame([
    {"Outcome": label, "Pairs": overall[key], "Share %": 100 * overall[key] / n}
    for label, key in [
        ("Both correct", "both_correct"), ("Text correct, image wrong", "text_only_correct"),
        ("Text wrong, image correct", "image_only_correct"), ("Both wrong", "both_wrong")
    ]
])
display(outcomes.style.format({"Share %": "{:.2f}"}))
retained = overall["both_correct"] / (overall["both_correct"] + overall["text_only_correct"])
print(f"Images retain {retained:.2%} of text-correct answers.")

## Answer-choice bias and response failures

Compare the answer distributions against the actual gold-label mix, shown above the heatmaps. Concentration on a choice is especially informative when accuracy approaches a constant-answer baseline. The `Invalid` column includes truncated responses, including any truncated single letter.

In [ ]:
show_plot(plot_answer_bias(report))
print(f"Always choosing the majority gold label: {report['majority_answer_baseline_pct']:.2f}% accuracy.")
print("Uniform random guessing: 25% expected accuracy.")
invalid_table = pd.DataFrame(report["invalid_by_language_mode"])
invalid_table["Language"] = invalid_table.language.map(LANGUAGE_NAMES)
display(invalid_table.loc[(invalid_table.invalid > 0) | (invalid_table.truncated > 0),
    ["Language", "mode", "n", "invalid", "truncated"]])
print("Invalid and truncated counts overlap; do not add them together.")
if report["invalid_examples"]:
    display(pd.DataFrame(report["invalid_examples"])[["language", "mode", "raw_response", "truncated"]])

## Processor resizing, page height, and multilingual consistency

The fixed **280-soft-token cap** compresses tall documents more. The first panel shows a geometric proxy: configured 24 px font size × the smaller processor scale. It is **not measured glyph height**. Actual soft-token counts may be below the cap because of patch-grid rounding.

Height quartiles are calculated separately within each language, keeping ties together. Their sizes can differ. These comparisons are descriptive: taller pages can also differ in content and difficulty, so the plot does not isolate a resolution effect. The last panel counts how many translations of each aligned question were answered correctly.

In [ ]:
show_plot(plot_diagnostics(report))
resize_table = pd.DataFrame(report["resizing"])
resize_table.insert(0, "Language", resize_table.language.map(LANGUAGE_NAMES))
display(resize_table.drop(columns="language").round(2))

# Per-language quartile counts make the height bins auditable.
height_table = pd.DataFrame(report["within_language_height_quartiles"])
display(height_table.pivot(index="language", columns="height_quartile", values="n").rename(
    index=LANGUAGE_NAMES, columns={1: "Q1 shorter", 2: "Q2", 3: "Q3", 4: "Q4 taller"}))
# Compare the shortest/tallest groups in both modalities, so difficulty is visible too.
ends = height_table[height_table.height_quartile.isin([1, 4])].copy()
ends["Language"] = ends.language.map(LANGUAGE_NAMES)
display(ends[["Language", "height_quartile", "n", "text_accuracy_pct", "image_accuracy_pct", "gap_pp"]].round(2))


## Interpretation and the next controlled experiment

Read the current run's accuracy, paired outcomes, answer distributions, and invalid-response counts above. The gap measures the full path from rendering and image preprocessing through visual encoding and answer generation. It does not isolate OCR, script, resolution, reasoning, or quantization as the cause.

A useful next experiment would keep IDs, checkpoint, prompts, answer order, and rendering fixed while increasing the image budget to 560 or 1120 in a new directory, after checking GPU fit. This notebook launches no evaluation and makes no cross-run comparison.



## Runtime and provenance

Generation latencies exclude loading and preparation and apply to this recorded hardware/software run.

In [ ]:
runtime = pd.DataFrame({mode: values for mode, values in report["runtime"].items()
                        if isinstance(values, dict)}).T
runtime["Generation minutes"] = runtime.latency_sum_seconds / 60
display(runtime[["Generation minutes", "latency_median_seconds", "latency_p95_seconds"]].round(3))
print("Model:", report["experiment"]["model_id"])
print("Model revision:", report["experiment"]["model_revision"])
print("Dataset revision:", report["experiment"]["dataset_revision"])
print("Manifest fingerprint:", report["manifest_fingerprint"])
# File hashes identify the original input artifacts; this notebook never rewrites them.
display(pd.Series(report["audit"]["original_artifact_sha256"], name="SHA-256").to_frame())

## Clear plot outputs before saving or sharing

Set `CLEAR_PLOTS = True` and run the next cell to replace the chart displays with text. To clear **all** saved outputs after saving, run:

```bash
.venv/bin/python scripts/clean_notebook.py Belebele_analysis.ipynb
```

Notebook autosave may persist outputs while viewing, so disable it if needed. No chart files or rendered images are created by the analysis. [README](README.md) records model/dataset/renderer attribution and evaluation details.

In [ ]:
CLEAR_PLOTS = False  # Set True before saving a clean notebook.
if CLEAR_PLOTS:
    for handle in plot_handles:
        handle.update("Plot cleared; rerun its cell to display it again.")
    plot_handles.clear()
    plt.close("all")
    print("Plot displays cleared.")